# Notebook 6: Bilevel Optimization — BRP vs Prosumer Conflict

## References

1. **Tushar, W., Yuen, C., Mohsenian-Rad, H., et al. (2018).** *"Transforming energy networks via peer-to-peer energy trading: the potential of game-theoretic approaches."* IEEE Signal Processing Magazine, 35(4), 90–111. [DOI: 10.1109/MSP.2018.2818327](https://doi.org/10.1109/MSP.2018.2818327)

2. **Saad, W., Han, Z., Poor, H. V., Başar, T. (2012).** *"Game-theoretic methods for the smart grid: An overview of microgrid systems, demand-side management, and smart grid communications."* IEEE Signal Processing Magazine, 29(5), 86–105. [DOI: 10.1109/MSP.2012.2186410](https://doi.org/10.1109/MSP.2012.2186410)

3. **Wei, W., Liu, F., Mei, S. (2015).** *"Energy pricing and dispatch for smart grid retailers under demand response and market price uncertainty."* IEEE Transactions on Smart Grid, 6(3), 1364–1374. [DOI: 10.1109/TSG.2014.2376522](https://doi.org/10.1109/TSG.2014.2376522)

4. **Zugno, M., Morales, J. M., Pinson, P., Madsen, H. (2013).** *"A bilevel model for electricity retailers' participation in a demand response market environment."* Energy Economics, 36, 182–197. [DOI: 10.1016/j.eneco.2012.12.010](https://doi.org/10.1016/j.eneco.2012.12.010)

## What these papers bring

**Tushar et al. (2018)** survey game-theoretic approaches for energy trading in prosumer networks, including Stackelberg games, Nash equilibrium, and cooperative game theory.

**Wei et al. (2015)** and **Zugno et al. (2013)** formulate bilevel optimization models where the retailer/BRP sets prices (upper level) and prosumers respond optimally (lower level). This is the **Stackelberg game** formulation.

## What is implemented below

We model the BRP–Prosumer interaction as a **Stackelberg bilevel optimization**:

- **Upper level (BRP)**: sets the internal tariff $\tau_t$ (buy/sell prices offered to prosumers)
- **Lower level (Prosumer)**: each prosumer optimally schedules their BESS given $\tau_t$

The BRP earns the **spread** between market prices and internal tariffs, plus benefits from portfolio netting. But if tariffs are too aggressive, prosumers leave (participation constraint). This captures the fundamental conflict described in the thesis goal.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

np.random.seed(42)


## 1. Problem Setup

### Players
- **BRP (leader)**: sets internal buy/sell tariffs $\tau^{buy}_t, \tau^{sell}_t$ for prosumers
- **Prosumers (followers)**: $N$ sites, each optimizes BESS given $\tau$
- **Market**: DA prices $c^{DA}_t$ are given

### BRP's revenue model
The BRP:
1. Buys electricity from the market at $c^{DA}_t$
2. Sells it to prosumers at $\tau^{buy}_t \geq c^{DA}_t$ (markup)
3. Buys surplus from prosumers at $\tau^{sell}_t \leq c^{DA}_t$ (discount)
4. BRP profit = prosumer payments − market costs − imbalance costs

### Prosumer's participation constraint
A prosumer stays in the BRP only if their cost under $\tau$ is **less** than their cost under their current supplier. We model the outside option as the prosumer's cost with buy/sell spread from the retail market.


In [ ]:
T = 24
hours = np.arange(T)
N = 5

# Market prices
c_da = np.maximum(0.04 + 0.02*np.sin(2*np.pi*(hours-6)/24) + 0.01*np.exp(-0.5*((hours-18)/3)**2), 0.015)

# Current supplier's tariffs (the "outside option")
# Typical retail: ~30% markup on DA for buy, ~40% discount for sell
tau_retail_buy = c_da * 1.30
tau_retail_sell = c_da * 0.60

# Site data
site_pv = []
site_load = []
site_flex = []  # flexibility (kW)
site_alpha = []  # discomfort weight

for i in range(N):
    pv_peak = 3.0 + 3.0 * np.random.rand()
    load_base = 1.5 + 2.0 * np.random.rand()
    pv = np.maximum(0, pv_peak * np.exp(-0.5*((hours-12)/3)**2))
    load = load_base + 0.8*np.exp(-0.5*((hours-8)/2)**2) + 1.2*np.exp(-0.5*((hours-19)/2.5)**2)
    site_pv.append(pv)
    site_load.append(load)
    site_flex.append(2.0 + 3.0 * np.random.rand())
    site_alpha.append(0.3 + 0.7 * np.random.rand())

# Net load (preferred exchange without battery)
site_d = [site_load[i] - site_pv[i] for i in range(N)]

print(f"Market DA price range: [{c_da.min()*1000:.1f}, {c_da.max()*1000:.1f}] EUR/MWh")
print(f"Retail buy range:      [{tau_retail_buy.min()*1000:.1f}, {tau_retail_buy.max()*1000:.1f}] EUR/MWh")
print(f"Retail sell range:     [{tau_retail_sell.min()*1000:.1f}, {tau_retail_sell.max()*1000:.1f}] EUR/MWh")


## 2. Lower Level: Prosumer Best Response

Given tariffs $\tau^{buy}_t, \tau^{sell}_t$, each prosumer minimizes:

$$\min_{g_i} \sum_t \left[\tau^{buy}_t \max(g_i(t), 0) - \tau^{sell}_t \max(-g_i(t), 0) + \frac{\alpha_i}{2}(g_i(t) - d_i(t))^2\right]$$

s.t. $d_i(t) - f_i \leq g_i(t) \leq d_i(t) + f_i$

For the bilevel formulation, we approximate $\max(g, 0) \approx \frac{g + \sqrt{g^2 + \epsilon}}{2}$ to make it smooth.


In [ ]:
eps_smooth = 0.01  # smoothing parameter

def smooth_pos(x):
    """Smooth approximation of max(x, 0)."""
    return (x + np.sqrt(x**2 + eps_smooth)) / 2

def smooth_pos_grad(x):
    """Gradient of smooth_pos."""
    return (1 + x / np.sqrt(x**2 + eps_smooth)) / 2

def prosumer_cost(g_i, d_i, flex_i, alpha_i, tau_buy, tau_sell):
    """Prosumer's cost given tariffs."""
    buy_cost = np.sum(tau_buy * smooth_pos(g_i))
    sell_rev = np.sum(tau_sell * smooth_pos(-g_i))
    discomfort = 0.5 * alpha_i * np.sum((g_i - d_i)**2)
    return buy_cost - sell_rev + discomfort

def prosumer_best_response(d_i, flex_i, alpha_i, tau_buy, tau_sell):
    """Solve prosumer's optimization given tariffs."""
    def obj(g):
        return prosumer_cost(g, d_i, flex_i, alpha_i, tau_buy, tau_sell)
    
    def grad(g):
        gr = tau_buy * smooth_pos_grad(g) + tau_sell * smooth_pos_grad(-g) + alpha_i * (g - d_i)
        return gr
    
    bounds = [(d_i[t]-flex_i, d_i[t]+flex_i) for t in range(len(d_i))]
    g0 = d_i.copy()
    res = minimize(obj, g0, jac=grad, bounds=bounds, method='L-BFGS-B')
    return res.x, res.fun

# Test: prosumer response to retail tariffs
print("Prosumer costs at RETAIL tariffs (outside option):")
retail_costs = []
for i in range(N):
    g_i, cost_i = prosumer_best_response(site_d[i], site_flex[i], site_alpha[i],
                                          tau_retail_buy, tau_retail_sell)
    retail_costs.append(cost_i)
    print(f"  Site {i}: {cost_i:.4f} EUR/day")


## 3. Upper Level: BRP Tariff Optimization

The BRP chooses $\tau^{buy}_t, \tau^{sell}_t$ to maximize profit:

$$\max_{\tau} \; \underbrace{\sum_i \sum_t \left[\tau^{buy}_t \max(g_i^*(t), 0) - \tau^{sell}_t \max(-g_i^*(t), 0)\right]}_{\text{prosumer payments}} - \underbrace{\sum_t c^{DA}_t \cdot z(t)}_{\text{market cost}}$$

where $g_i^*(\tau)$ is each prosumer's optimal response to tariff $\tau$, and $z(t) = \sum_i g_i^*(t)$.

**Participation constraint**: $\text{cost}_i(\tau) \leq \text{cost}_i(\tau^{retail}) - \delta$ — prosumers must save at least $\delta$ compared to their current supplier.


In [ ]:
delta_saving = 0.01  # minimum savings (EUR/day) for prosumer participation

def evaluate_tariff(tau_flat):
    """Given tariff parameters, compute BRP profit and prosumer costs."""
    tau_buy = tau_flat[:T]
    tau_sell = tau_flat[T:]
    
    # Prosumer responses
    total_prosumer_payment = 0
    prosumer_costs = []
    portfolio_exchange = np.zeros(T)
    
    for i in range(N):
        g_i, cost_i = prosumer_best_response(site_d[i], site_flex[i], site_alpha[i],
                                              tau_buy, tau_sell)
        # Prosumer pays BRP
        payment_i = np.sum(tau_buy * smooth_pos(g_i) - tau_sell * smooth_pos(-g_i))
        total_prosumer_payment += payment_i
        prosumer_costs.append(cost_i)
        portfolio_exchange += g_i
    
    # BRP's market cost
    market_cost = np.sum(c_da * portfolio_exchange)
    
    # BRP profit
    brp_profit = total_prosumer_payment - market_cost
    
    # Participation: all prosumers must save vs retail
    participation_penalty = 0
    for i in range(N):
        shortfall = prosumer_costs[i] - (retail_costs[i] - delta_saving)
        if shortfall > 0:
            participation_penalty += 100 * shortfall  # heavy penalty
    
    return brp_profit, prosumer_costs, participation_penalty

def brp_objective(tau_flat):
    """BRP maximizes profit (we minimize negative profit + participation penalty)."""
    profit, costs, penalty = evaluate_tariff(tau_flat)
    return -profit + penalty

# Tariff constraints: c_da <= tau_buy, tau_sell <= c_da (basic ordering)
# Also: tau_sell < tau_buy (buy price > sell price)
tau0_buy = c_da * 1.15  # start at 15% markup
tau0_sell = c_da * 0.75  # start at 25% discount
tau0 = np.concatenate([tau0_buy, tau0_sell])

# Bounds
bounds = []
for t in range(T):
    bounds.append((c_da[t] * 1.01, c_da[t] * 1.50))  # tau_buy: at least DA price, max 50% markup
for t in range(T):
    bounds.append((c_da[t] * 0.40, c_da[t] * 0.99))  # tau_sell: max DA price, min 40% of DA

print("Optimizing BRP tariffs (Stackelberg bilevel)...")
result = minimize(brp_objective, tau0, bounds=bounds, method='L-BFGS-B',
                  options={'maxiter': 200, 'ftol': 1e-8})

tau_opt_buy = result.x[:T]
tau_opt_sell = result.x[T:]
profit_opt, costs_opt, penalty_opt = evaluate_tariff(result.x)

print(f"Optimization status: {result.message}")
print(f"BRP profit: {profit_opt:.4f} EUR/day")
print(f"Participation penalty: {penalty_opt:.4f} (should be 0)")
print()
print(f"{'Site':<6} {'Retail cost':<14} {'BRP cost':<14} {'Saving':<10} {'Saving%':<8}")
print("-"*50)
for i in range(N):
    saving = retail_costs[i] - costs_opt[i]
    print(f"  {i:<4} {retail_costs[i]:10.4f}    {costs_opt[i]:10.4f}    {saving:8.4f}  {100*saving/max(abs(retail_costs[i]),0.01):6.1f}%")


In [ ]:
# Compare tariff structures
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Tariff comparison
axes[0,0].fill_between(hours, tau_retail_sell*1000, tau_retail_buy*1000, alpha=0.1, color='gray', label='Retail spread')
axes[0,0].plot(hours, tau_retail_buy*1000, 'gray', ls='--', lw=1, label='Retail buy')
axes[0,0].plot(hours, tau_retail_sell*1000, 'gray', ls=':', lw=1, label='Retail sell')
axes[0,0].plot(hours, tau_opt_buy*1000, 'r-o', ms=3, lw=2, label='BRP τ_buy')
axes[0,0].plot(hours, tau_opt_sell*1000, 'g-s', ms=3, lw=2, label='BRP τ_sell')
axes[0,0].plot(hours, c_da*1000, 'b--', lw=1.5, label='DA price')
axes[0,0].set_xlabel('Hour'); axes[0,0].set_ylabel('EUR/MWh')
axes[0,0].set_title('Tariff Comparison: BRP vs Retail vs DA')
axes[0,0].legend(fontsize=6); axes[0,0].grid(alpha=0.3)

# Prosumer cost comparison
x_pos = np.arange(N)
w = 0.35
bars1 = axes[0,1].bar(x_pos-w/2, retail_costs, w, label='Retail', color='gray', alpha=0.7)
bars2 = axes[0,1].bar(x_pos+w/2, costs_opt, w, label='BRP', color='steelblue', alpha=0.7)
axes[0,1].set_xlabel('Site'); axes[0,1].set_ylabel('EUR/day')
axes[0,1].set_title('Prosumer Daily Cost: Retail vs BRP')
axes[0,1].set_xticks(x_pos); axes[0,1].legend(); axes[0,1].grid(alpha=0.3, axis='y')

# BRP markup/discount profile
markup = (tau_opt_buy - c_da) / c_da * 100
discount = (c_da - tau_opt_sell) / c_da * 100
axes[1,0].bar(hours-0.2, markup, 0.4, color='coral', alpha=0.7, label='Buy markup %')
axes[1,0].bar(hours+0.2, discount, 0.4, color='teal', alpha=0.7, label='Sell discount %')
axes[1,0].set_xlabel('Hour'); axes[1,0].set_ylabel('%')
axes[1,0].set_title('BRP Markup/Discount over DA Price')
axes[1,0].legend(); axes[1,0].grid(alpha=0.3, axis='y')

# Profit decomposition
# Compute portfolio exchange at optimal tariff
g_opt_all = np.zeros((N, T))
for i in range(N):
    g_i, _ = prosumer_best_response(site_d[i], site_flex[i], site_alpha[i],
                                     tau_opt_buy, tau_opt_sell)
    g_opt_all[i] = g_i
z_opt = np.sum(g_opt_all, axis=0)

hourly_revenue = np.zeros(T)
for i in range(N):
    hourly_revenue += tau_opt_buy * smooth_pos(g_opt_all[i]) - tau_opt_sell * smooth_pos(-g_opt_all[i])
hourly_market = c_da * z_opt
hourly_profit = hourly_revenue - hourly_market

axes[1,1].bar(hours, hourly_profit*1000, color=['teal' if p>0 else 'coral' for p in hourly_profit], alpha=0.7)
axes[1,1].set_xlabel('Hour'); axes[1,1].set_ylabel('EUR/MWh equiv')
axes[1,1].set_title(f'Hourly BRP Profit (Total: {profit_opt:.3f} EUR/day)')
axes[1,1].grid(alpha=0.3, axis='y')

plt.tight_layout(); plt.savefig('/tmp/nb6_results.png', dpi=100); plt.show()


## 4. Sensitivity: Prosumer Minimum Saving Requirement


In [ ]:
deltas = [0.0, 0.02, 0.05, 0.10, 0.20, 0.50]
brp_profits = []
avg_savings = []

for delta in deltas:
    delta_saving_test = delta
    
    def brp_obj_test(tau_flat):
        tau_b, tau_s = tau_flat[:T], tau_flat[T:]
        total_payment = 0; costs_list = []; port = np.zeros(T)
        for i in range(N):
            g_i, c_i = prosumer_best_response(site_d[i], site_flex[i], site_alpha[i], tau_b, tau_s)
            total_payment += np.sum(tau_b*smooth_pos(g_i) - tau_s*smooth_pos(-g_i))
            costs_list.append(c_i); port += g_i
        mkt = np.sum(c_da * port)
        profit = total_payment - mkt
        pen = sum(max(0, c_i - (retail_costs[i] - delta_saving_test))*100 
                  for i, c_i in enumerate(costs_list))
        return -profit + pen
    
    res = minimize(brp_obj_test, tau0, bounds=bounds, method='L-BFGS-B',
                   options={'maxiter': 200, 'ftol': 1e-8})
    tau_b, tau_s = res.x[:T], res.x[T:]
    
    profit = 0; savings = []
    for i in range(N):
        g_i, c_i = prosumer_best_response(site_d[i], site_flex[i], site_alpha[i], tau_b, tau_s)
        profit += np.sum(tau_b*smooth_pos(g_i) - tau_s*smooth_pos(-g_i))
        savings.append(retail_costs[i] - c_i)
        profit -= np.sum(c_da * g_i)  # market cost (per-site for simplicity)
    
    brp_profits.append(profit)
    avg_savings.append(np.mean(savings))

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax2 = ax.twinx()
ax.bar([f'{d:.2f}' for d in deltas], brp_profits, color='steelblue', alpha=0.7, label='BRP profit')
ax2.plot([f'{d:.2f}' for d in deltas], [s*100/np.mean(retail_costs) for s in avg_savings],
         'ro-', lw=2, label='Avg prosumer saving %')
ax.set_xlabel('Min saving δ (EUR/day)')
ax.set_ylabel('BRP profit (EUR/day)', color='steelblue')
ax2.set_ylabel('Avg prosumer saving (%)', color='red')
ax.set_title('Trade-off: BRP Profit vs Prosumer Satisfaction')
ax.legend(loc='upper left'); ax2.legend(loc='upper right')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.savefig('/tmp/nb6_tradeoff.png', dpi=100); plt.show()

print(f"\n{'δ':>6} {'BRP profit':>12} {'Avg saving':>12}")
print("-"*32)
for d, p, s in zip(deltas, brp_profits, avg_savings):
    print(f"{d:6.2f} {p:12.4f} {s:12.4f}")


## 5. Cooperative Game Theory: Benefit Sharing

An alternative to Stackelberg pricing is **cooperative game theory** using the **Shapley value** or **nucleolus** to distribute the coalition surplus fairly.

The **coalition surplus** = (cost of all prosumers independently) − (cost of the BRP portfolio):


In [ ]:
# Individual costs (each prosumer alone, retail tariff)
individual_costs = np.array(retail_costs)
total_individual = np.sum(individual_costs)

# Coalition cost: centralized portfolio at DA price (no spread)
from scipy.optimize import minimize as sp_minimize

def coalition_cost(g_flat):
    G = g_flat.reshape(N, T)
    z = np.sum(G, axis=0)
    # Portfolio buys/sells at DA price (no spread for BRP)
    market = np.sum(c_da * z)
    discomfort = sum(0.5*site_alpha[i]*np.sum((G[i]-site_d[i])**2) for i in range(N))
    return market + discomfort

bnds = []
for i in range(N):
    for t in range(T):
        bnds.append((site_d[i][t]-site_flex[i], site_d[i][t]+site_flex[i]))
g0 = np.concatenate(site_d)
res_coal = sp_minimize(coalition_cost, g0, bounds=bnds, method='L-BFGS-B')
total_coalition = res_coal.fun

surplus = total_individual - total_coalition
print(f"Total individual cost (retail): {total_individual:.4f} EUR/day")
print(f"Coalition cost (BRP portfolio): {total_coalition:.4f} EUR/day")
print(f"Coalition surplus:              {surplus:.4f} EUR/day")
print()

# Simple proportional sharing: each gets saving proportional to their contribution
G_coal = res_coal.x.reshape(N, T)
z_coal = np.sum(G_coal, axis=0)

# Shapley-like allocation: split surplus proportionally to individual cost
print(f"{'Site':<6} {'Indep cost':<12} {'Share%':<8} {'Allocation':<12} {'Net cost':<12} {'Saving':<10}")
print("-"*60)
for i in range(N):
    share = individual_costs[i] / total_individual
    allocation = share * surplus
    net = individual_costs[i] - allocation
    saving_pct = 100 * allocation / max(individual_costs[i], 0.01)
    print(f"  {i:<4} {individual_costs[i]:10.4f}  {100*share:6.1f}  {allocation:10.4f}  {net:10.4f}  {saving_pct:8.1f}%")

brp_share = 0.3  # BRP keeps 30% of surplus
print(f"\nWith BRP keeping {100*brp_share:.0f}% of surplus:")
print(f"  BRP income:  {brp_share * surplus:.4f} EUR/day")
print(f"  Prosumer pool: {(1-brp_share) * surplus:.4f} EUR/day")
for i in range(N):
    share = individual_costs[i] / total_individual
    alloc = share * (1-brp_share) * surplus
    print(f"  Site {i}: saves {alloc:.4f} EUR/day ({100*alloc/max(individual_costs[i],0.01):.1f}%)")


## 6. Key Insights for the Thesis

### BRP–Prosumer Conflict and Resolution

1. **The fundamental conflict**: the BRP wants to maximize its profit (wider spread), while prosumers want lower costs (narrower spread). The bilevel optimization captures this as a Stackelberg game.

2. **Participation constraint is key**: if the BRP's tariffs are too aggressive, prosumers leave for a retail supplier. The minimum saving $\delta$ constrains the BRP's pricing power.

3. **Optimal tariff structure**: the BRP's optimal strategy is to set tariffs that:
   - Are competitive with retail (satisfy participation)
   - Create portfolio netting value (some of the spread is recaptured at portfolio level)
   - Vary by time-of-day (higher markup at peak, lower off-peak)

4. **Cooperative vs competitive framing**: cooperative game theory (Shapley value) provides a **fairer** allocation than the Stackelberg model. In practice, a hybrid approach works:
   - Use cooperative allocation to set long-term tariff structure
   - Use competitive pricing for short-term adjustments

5. **Practical recommendation for the company**:
   - Start with a simple proportional surplus sharing (e.g., 70% to prosumers, 30% to BRP)
   - Monitor prosumer satisfaction and adjust the split
   - Use the bilevel model to find the optimal hourly tariff profile
   - The internal transfer price $\mu^*$ from Notebook 4 provides an alternative pricing mechanism

6. **Czech/Slovak regulatory context**: the BRP must provide transparent pricing. The Shapley value or proportional sharing provides a defensible, fair mechanism.

---

### Summary of All Notebooks

| Notebook | Approach | Contribution |
|----------|----------|-------------|
| 1 | Single-site MILP | Current company approach (baseline) |
| 2 | Centralized portfolio MILP | Multi-site coordination, netting benefit |
| 3 | Two-stage stochastic | Handling forecast uncertainty, risk |
| 4 | ADMM decomposition | Privacy-preserving distributed coordination |
| 5 | MPC real-time | Nomination tracking, imbalance reduction |
| 6 | Bilevel / Game theory | BRP–prosumer conflict, fair pricing |
